# MedXAI — Notebook 03 : Comparaison XAI
## ACF+Anchor vs Bio_ClinicalBERT (Captum + BERTViz)

**Objectif :** Comparer les explications produites par deux approches :
- **ACF+Anchor** : règles logiques mathématiques (notre algorithme original)
- **Captum** : attribution de features sur Bio_ClinicalBERT (approche neuronale)

**Question centrale :** Les deux méthodes identifient-elles les mêmes symptômes décisifs ?

In [4]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import IntegratedGradients
from sklearn.preprocessing import LabelEncoder

sys.path.append(os.path.abspath('..'))
from src.acf_das import FormalContext
from src.anchor_rules import generate_rules

print(f"Imports OK")
print(f"PyTorch : {torch.__version__}")

Imports OK
PyTorch : 2.13.0+cpu


In [5]:

# CHARGEMENT DATASET + MODÈLES

# Dataset
df = pd.read_csv('data/dataset.csv')
maladies = ['Bronchial Asthma', 'Tuberculosis', 'Pneumonia', 'Common Cold']
df_resp = df[df['Disease'].isin(maladies)].reset_index(drop=True)

# Label encoder
le = LabelEncoder()
le.fit(df_resp['Disease'])

# Contexte formel + DAS
fc = FormalContext(df_resp)
binary_matrix = fc.build_context()
das_dict = {}
for disease in maladies:
    das_dict[disease] = fc.compute_das(disease)

# Règles Anchor
df_rules = generate_rules(das_dict, binary_matrix)

# Charger Bio_ClinicalBERT fine-tuné
model_path = '../results/clinical_bert/final_model'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

print("Dataset, ACF et modèle chargés")
print(f"Classes : {le.classes_}")

Construction du contexte formel...
Contexte formel construit :
   480 patients
   29 symptômes uniques
   4 maladies

Calcul des DAS pour : Bronchial Asthma
6 DAS trouvés pour Bronchial Asthma

Calcul des DAS pour : Tuberculosis
16 DAS trouvés pour Tuberculosis

Calcul des DAS pour : Pneumonia
11 DAS trouvés pour Pneumonia

Calcul des DAS pour : Common Cold
17 DAS trouvés pour Common Cold
Génération des règles Anchor...

--- Bronchial Asthma ---
  R1: IF breathlessness THEN Bronchial Asthma
       Couverture = 0.95
  R2: IF cough THEN Bronchial Asthma
       Couverture = 0.9
  R3: IF family_history THEN Bronchial Asthma
       Couverture = 0.95
  R4: IF fatigue THEN Bronchial Asthma
       Couverture = 0.9
  R5: IF high_fever THEN Bronchial Asthma
       Couverture = 0.95
  R6: IF mucoid_sputum THEN Bronchial Asthma
       Couverture = 0.95
--- Tuberculosis ---
  R1: IF blood_in_sputum THEN Tuberculosis
       Couverture = 1.0
  R2: IF breathlessness THEN Tuberculosis
       Couverture

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Dataset, ACF et modèle chargés
Classes : ['Bronchial Asthma' 'Common Cold' 'Pneumonia' 'Tuberculosis']


In [6]:

# FONCTION XAI CAPTUM — attribution par Integrated Gradients

def get_captum_attributions(text, model, tokenizer, label_idx):
    """
    Calcule l'importance de chaque token pour la prédiction
    via Integrated Gradients (Captum).
    """
    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    )

    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    # Embedding layer
    embeddings = model.bert.embeddings(input_ids)

    def forward_func(embeddings):
        outputs = model(
            inputs_embeds=embeddings,
            attention_mask=attention_mask
        )
        return outputs.logits

    # Integrated Gradients
    ig = IntegratedGradients(forward_func)
    attributions = ig.attribute(
        embeddings,
        target=label_idx,
        n_steps=30
    )

    # Score par token
    scores = attributions.squeeze(0).sum(dim=-1).detach().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    # Filtrer les tokens spéciaux
    result = [
        (tok.replace('##', ''), float(sc))
        for tok, sc in zip(tokens, scores)
        if tok not in ['[CLS]', '[SEP]', '[PAD]']
    ]

    return result

print("Fonction Captum prête")

Fonction Captum prête


In [11]:

# TEST CAPTUM SUR UN PATIENT RÉEL — Tuberculosis

# Convertir les symptômes en texte
symptom_cols = [c for c in df_resp.columns if c.startswith('Symptom_')]

def symptoms_to_text(row):
    symptoms = [row[col].strip() for col in symptom_cols if pd.notna(row[col])]
    return f"Patient presents with : {', '.join(symptoms)}"

# Prendre un patient Tuberculosis
patient_tb = df_resp[df_resp['Disease'] == 'Tuberculosis'].iloc[0]
text_tb = symptoms_to_text(patient_tb)
label_idx = int(le.transform(['Tuberculosis'])[0])

print(f"Patient : {text_tb}")
print(f"Diagnostic réel : Tuberculosis (label={label_idx})\n")

# Captum
attributions = get_captum_attributions(text_tb, model, tokenizer, label_idx)

# Trier par importance
attributions_sorted = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)

print("Top 10 tokens importants pour Bio_ClinicalBERT :")
for token, score in attributions_sorted[:10]:
    bar = "█" * int(abs(score) * 20)
    print(f"  {token:<25} {score:+.4f} {bar}")

Patient : Patient presents with : chills, vomiting, fatigue, weight_loss, cough, high_fever, breathlessness, sweating, loss_of_appetite, mild_fever, yellowing_of_eyes, swelled_lymph_nodes, malaise, phlegm, chest_pain, blood_in_sputum
Diagnostic réel : Tuberculosis (label=3)

Top 10 tokens importants pour Bio_ClinicalBERT :
  patient                   +0.3377 ██████
  fever                     +0.3149 ██████
  presents                  +0.2940 █████
  chill                     +0.2704 █████
  with                      +0.1882 ███
  lai                       +0.1680 ███
  blood                     +0.1596 ███
  in                        +0.1568 ███
  fever                     +0.1542 ███
  appetite                  +0.1406 ██


In [12]:
# ============================================================
# RECONSTRUCTION DES SYMPTÔMES DEPUIS LES TOKENS BERT
# ============================================================

def get_symptom_attributions(text, model, tokenizer, label_idx):
    """
    Calcule l'importance par SYMPTÔME complet
    en fusionnant les tokens BERT découpés.
    """
    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    )

    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']
    embeddings = model.bert.embeddings(input_ids)

    def forward_func(embeddings):
        outputs = model(
            inputs_embeds=embeddings,
            attention_mask=attention_mask
        )
        return outputs.logits

    ig = IntegratedGradients(forward_func)
    attributions = ig.attribute(
        embeddings,
        target=label_idx,
        n_steps=30
    )

    scores = attributions.squeeze(0).sum(dim=-1).detach().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    # Fusionner les tokens ## pour reconstruire les mots complets
    merged_tokens = []
    merged_scores = []
    current_word = ""
    current_score = 0.0

    for tok, sc in zip(tokens, scores):
        if tok in ['[CLS]', '[SEP]', '[PAD]', ',', ':']:
            if current_word:
                merged_tokens.append(current_word)
                merged_scores.append(current_score)
                current_word = ""
                current_score = 0.0
        elif tok.startswith('##'):
            current_word += tok[2:]
            current_score += float(sc)
        else:
            if current_word:
                merged_tokens.append(current_word)
                merged_scores.append(current_score)
            current_word = tok
            current_score = float(sc)

    if current_word:
        merged_tokens.append(current_word)
        merged_scores.append(current_score)

    # Filtrer les mots non pertinents
    stop_words = ['patient', 'presents', 'with', 'and', 'the', 'a']
    result = [
        (tok, sc) for tok, sc in zip(merged_tokens, merged_scores)
        if tok.lower() not in stop_words and len(tok) > 2
    ]

    return sorted(result, key=lambda x: abs(x[1]), reverse=True)


# Test sur Tuberculosis
patient_tb = df_resp[df_resp['Disease'] == 'Tuberculosis'].iloc[0]
text_tb = symptoms_to_text(patient_tb)
label_idx = int(le.transform(['Tuberculosis'])[0])

attrs = get_symptom_attributions(text_tb, model, tokenizer, label_idx)

print("Top 10 symptômes importants pour Bio_ClinicalBERT — Tuberculosis :\n")
for token, score in attrs[:10]:
    bar = "█" * int(abs(score) * 15)
    print(f"  {token:<25} {score:+.4f} {bar}")

print("\nDAS ACF+Anchor pour Tuberculosis :")
for das in das_dict['Tuberculosis'][:10]:
    print(f"  - {das[0].strip()}")

Top 10 symptômes importants pour Bio_ClinicalBERT — Tuberculosis :

  sputum                    +0.3601 █████
  chills                    +0.3547 █████
  malaise                   +0.3322 ████
  fever                     +0.3149 ████
  phlegm                    +0.2092 ███
  yellowing                 +0.1794 ██
  lymph                     +0.1659 ██
  blood                     +0.1596 ██
  fever                     +0.1542 ██
  appetite                  +0.1406 ██

DAS ACF+Anchor pour Tuberculosis :
  - blood_in_sputum
  - breathlessness
  - chest_pain
  - chills
  - cough
  - fatigue
  - high_fever
  - loss_of_appetite
  - malaise
  - mild_fever
